# Model Training, Experiments, Tuning & Evaluation — Olist Delivery Time Prediction

Tuân thủ chặt chẽ quy trình chống rò rỉ dữ liệu:
- **Test set được khóa hoàn toàn** trong suốt các bước chọn encoding, Feature Ablation (bao gồm `Without_TE` và `With_Olist_ETA`), `TimeSeriesSplit` Tuning, kiểm tra ổn định và Permutation Importance.
- Đánh giá **Test set 1 lần duy nhất** ở Mục 6 + phân tích lỗi theo nhóm + Top 5 Worst Errors.
- Bổ sung ở Mục 8: **Sensitivity Analysis** (Right-censoring Cutoff `<= 2018-06-30` & Outlier Cap `p99`), **Quantile Regression (P10–P90)**, và biểu đồ **MAE theo tháng trên Val & Test**.

In [9]:
import json
import os
import warnings
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
import lightgbm as lgb
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.dpi'] = 120

PROC_DIR = 'processed'
MODEL_DIR = 'model'
FIG_DIR = 'model_figures'
os.makedirs(MODEL_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

## 1. Load dữ liệu & Hàm đánh giá

In [10]:
with open(os.path.join(PROC_DIR, 'feature_metadata.json'), 'r', encoding='utf-8') as f:
    meta = json.load(f)

target_col = meta['target_col']
bench_col = meta['benchmark_col']
num_features = meta['num_features']
cat_features = meta['cat_features']
all_features = num_features + cat_features

train_df = pd.read_csv(os.path.join(PROC_DIR, 'train.csv'))
val_df = pd.read_csv(os.path.join(PROC_DIR, 'val.csv'))
test_df = pd.read_csv(os.path.join(PROC_DIR, 'test.csv'))

X_train, y_train = train_df[all_features], train_df[target_col].values
X_val, y_val = val_df[all_features], val_df[target_col].values
X_test, y_test = test_df[all_features], test_df[target_col].values

def eval_metrics(y_true, y_pred):
    err = np.abs(y_true - y_pred)
    return {
        'MAE': float(mean_absolute_error(y_true, y_pred)),
        'RMSE': float(np.sqrt(mean_squared_error(y_true, y_pred))),
        'R2': float(r2_score(y_true, y_pred)),
        'Within_3d_%': float((err <= 3.0).mean() * 100.0),
        'Within_5d_%': float((err <= 5.0).mean() * 100.0),
    }

def make_preprocessor(num_cols, cat_cols, cat_encoding='onehot'):
    if cat_encoding == 'onehot':
        cat_tf = OneHotEncoder(handle_unknown='ignore', min_frequency=20, sparse_output=False)
        num_tf = StandardScaler()
    else:
        cat_tf = OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1)
        num_tf = 'passthrough'
    return ColumnTransformer([('num', num_tf, num_cols), ('cat', cat_tf, cat_cols)])

print(f'Prediction time: {meta["prediction_time"]} | Train={len(train_df):,}, Val={len(val_df):,}, Test={len(test_df):,} (LOCKED)')

Prediction time: purchase_time | Train=67,529, Val=14,470, Test=14,471 (LOCKED)


## 2. Thí nghiệm So sánh `OrdinalEncoder` vs `OneHotEncoder` trên tập Validation

In [11]:
enc_results = []
for enc in ['ordinal', 'onehot']:
    prep = make_preprocessor(num_features, cat_features, cat_encoding=enc)
    X_tr_e = prep.fit_transform(X_train)
    X_va_e = prep.transform(X_val)

    if enc == 'onehot':
        ridge = Ridge(alpha=10.0, random_state=42).fit(X_tr_e, y_train)
        m = eval_metrics(y_val, ridge.predict(X_va_e))
        enc_results.append({'Model': 'Ridge_Regression', 'Encoding': enc, 'Val_MAE': m['MAE'], 'Val_RMSE': m['RMSE'], 'Val_R2': m['R2']})

    xm = xgb.XGBRegressor(
        objective='reg:absoluteerror', n_estimators=600, learning_rate=0.05,
        max_depth=6, subsample=0.8, colsample_bytree=0.8, n_jobs=-1, random_state=42
    ).fit(X_tr_e, y_train)
    m = eval_metrics(y_val, xm.predict(X_va_e))
    enc_results.append({'Model': 'XGBoost_L1', 'Encoding': enc, 'Val_MAE': m['MAE'], 'Val_RMSE': m['RMSE'], 'Val_R2': m['R2']})

    lm = lgb.LGBMRegressor(
        objective='regression_l1', n_estimators=600, learning_rate=0.05,
        num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
        n_jobs=-1, random_state=42, verbose=-1
    ).fit(X_tr_e, y_train)
    m = eval_metrics(y_val, lm.predict(X_va_e))
    enc_results.append({'Model': 'LightGBM_L1', 'Encoding': enc, 'Val_MAE': m['MAE'], 'Val_RMSE': m['RMSE'], 'Val_R2': m['R2']})

enc_df = pd.DataFrame(enc_results).sort_values('Val_MAE')
enc_df.to_csv(os.path.join(MODEL_DIR, 'encoding_comparison.csv'), index=False)
best_tree_enc = enc_df[enc_df['Model'].isin(['XGBoost_L1', 'LightGBM_L1'])].iloc[0]['Encoding']
print(f'Selected tree categorical encoding (by Val MAE): {best_tree_enc}')
enc_df

Selected tree categorical encoding (by Val MAE): onehot


,Model,Encoding,Val_MAE,Val_RMSE,Val_R2
3,XGBoost_L1,onehot,4.123885,6.232746,0.347408
4,LightGBM_L1,onehot,4.127642,6.252665,0.343230
0,XGBoost_L1,ordinal,4.147076,6.235771,0.346774
1,LightGBM_L1,ordinal,4.161375,6.244291,0.344988
2,Ridge_Regression,onehot,4.727679,6.607077,0.266666


## 3. Thí nghiệm Feature Ablation mở rộng trên tập Validation (bao gồm `Without_TE` & `With_Olist_ETA`)

In [12]:
fs_A_num = ['purchase_hour', 'purchase_dayofweek', 'purchase_day', 'purchase_month', 'is_weekend', 'total_items', 'total_price', 'total_freight', 'num_unique_sellers']
fs_A_cat = ['customer_state', 'primary_seller_state']

fs_B_num = fs_A_num + ['num_unique_products', 'total_weight_g', 'max_weight_g', 'total_volume_cm3', 'volumetric_weight_g', 'missing_weight_count', 'missing_dimension_count', 'avg_photos_qty', 'freight_ratio', 'price_per_item', 'freight_per_kg']
fs_B_cat = fs_A_cat + ['primary_category']

fs_C_num = fs_B_num + ['haversine_km', 'cust_lat', 'cust_lng', 'sell_lat', 'sell_lng', 'is_same_state', 'is_same_city']
fs_C_cat = fs_B_cat

fs_no_te_num = [c for c in num_features if not c.endswith('_te')]
fs_eta_num = num_features + [bench_col]

ablation_sets = {
    'A_Basic': (fs_A_num, fs_A_cat),
    'B_Basic_Product': (fs_B_num, fs_B_cat),
    'C_Basic_Product_Geo': (fs_C_num, fs_C_cat),
    'D_Full_Valid_Features': (num_features, cat_features),
    'E_Without_TE': (fs_no_te_num, cat_features),
    'F_With_Olist_ETA_Calibration': (fs_eta_num, cat_features),
}

ablation_logs = []
for name, (n_cols, c_cols) in ablation_sets.items():
    prep = make_preprocessor(n_cols, c_cols, cat_encoding=best_tree_enc)
    X_tr_a = prep.fit_transform(train_df[n_cols + c_cols])
    X_va_a = prep.transform(val_df[n_cols + c_cols])
    m_lgb = lgb.LGBMRegressor(
        objective='regression_l1', n_estimators=600, learning_rate=0.05,
        num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1,
        n_jobs=-1, random_state=42, verbose=-1
    ).fit(X_tr_a, y_train)
    met = eval_metrics(y_val, m_lgb.predict(X_va_a))
    ablation_logs.append({'Feature_Set': name, 'Num_Features': len(n_cols) + len(c_cols), 'Val_MAE': met['MAE'], 'Val_RMSE': met['RMSE'], 'Val_R2': met['R2']})

ablation_df = pd.DataFrame(ablation_logs)
ablation_df.to_csv(os.path.join(MODEL_DIR, 'ablation_results.csv'), index=False)
print(ablation_df.to_string(index=False))

                 Feature_Set  Num_Features  Val_MAE  Val_RMSE   Val_R2
                     A_Basic            11 4.511937  6.726186 0.239987
             B_Basic_Product            23 4.347506  6.552983 0.278625
         C_Basic_Product_Geo            30 4.114488  6.343706 0.323965
       D_Full_Valid_Features            42 4.127642  6.252665 0.343230
                E_Without_TE            38 4.129915  6.373301 0.317643
F_With_Olist_ETA_Calibration            43 4.104059  6.230542 0.347869


## 4. `TimeSeriesSplit` Cross-Validation & Hyperparameter Tuning

In [13]:
tree_prep = make_preprocessor(num_features, cat_features, cat_encoding=best_tree_enc)
X_train_t = tree_prep.fit_transform(X_train)
X_val_t = tree_prep.transform(X_val)

tscv = TimeSeriesSplit(n_splits=5)
rng = np.random.RandomState(42)

def eval_tscv(model_fn):
    fold_maes, fold_rmses = [], []
    for tr_idx, va_idx in tscv.split(X_train_t):
        m = model_fn()
        m.fit(X_train_t[tr_idx], y_train[tr_idx])
        pred = m.predict(X_train_t[va_idx])
        fold_maes.append(mean_absolute_error(y_train[va_idx], pred))
        fold_rmses.append(np.sqrt(mean_squared_error(y_train[va_idx], pred)))
    return float(np.mean(fold_maes)), float(np.std(fold_maes)), float(np.mean(fold_rmses))

lgb_grid = {
    'num_leaves': [31, 63, 95],
    'min_child_samples': [30, 50, 80],
    'learning_rate': [0.03, 0.05],
    'colsample_bytree': [0.7, 0.8],
    'subsample': [0.8, 0.9],
    'reg_alpha': [0.0, 1.0],
    'reg_lambda': [1.0, 5.0],
}
lgb_cv_logs = []
for trial in range(6):
    p = {k: rng.choice(v).item() for k, v in lgb_grid.items()}
    mean_mae, std_mae, mean_rmse = eval_tscv(
        lambda p=p: lgb.LGBMRegressor(objective='regression_l1', n_estimators=600, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1, **p)
    )
    lgb_cv_logs.append({'Trial': trial + 1, 'CV_MAE_Mean': mean_mae, 'CV_MAE_Std': std_mae, 'CV_RMSE_Mean': mean_rmse, **p})

lgb_cv_df = pd.DataFrame(lgb_cv_logs).sort_values('CV_MAE_Mean')
best_lgb_params = lgb_cv_df.iloc[0][list(lgb_grid.keys())].to_dict()
best_lgb_params['num_leaves'] = int(best_lgb_params['num_leaves'])
best_lgb_params['min_child_samples'] = int(best_lgb_params['min_child_samples'])
print(f"Best LightGBM_L1 CV MAE: {lgb_cv_df.iloc[0]['CV_MAE_Mean']:.4f} ± {lgb_cv_df.iloc[0]['CV_MAE_Std']:.4f}")

xgb_grid = {
    'max_depth': [5, 6, 8],
    'learning_rate': [0.03, 0.05],
    'min_child_weight': [10, 20, 30],
    'subsample': [0.7, 0.8],
    'colsample_bytree': [0.7, 0.8],
    'reg_alpha': [0.0, 1.0],
    'reg_lambda': [1.0, 5.0],
}
xgb_cv_logs = []
for trial in range(6):
    p = {k: rng.choice(v).item() for k, v in xgb_grid.items()}
    mean_mae, std_mae, mean_rmse = eval_tscv(
        lambda p=p: xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=600, n_jobs=-1, random_state=42, **p)
    )
    xgb_cv_logs.append({'Trial': trial + 1, 'CV_MAE_Mean': mean_mae, 'CV_MAE_Std': std_mae, 'CV_RMSE_Mean': mean_rmse, **p})

xgb_cv_df = pd.DataFrame(xgb_cv_logs).sort_values('CV_MAE_Mean')
best_xgb_params = xgb_cv_df.iloc[0][list(xgb_grid.keys())].to_dict()
best_xgb_params['max_depth'] = int(best_xgb_params['max_depth'])
best_xgb_params['min_child_weight'] = int(best_xgb_params['min_child_weight'])
print(f"Best XGBoost_L1 CV MAE:  {xgb_cv_df.iloc[0]['CV_MAE_Mean']:.4f} ± {xgb_cv_df.iloc[0]['CV_MAE_Std']:.4f}")

lgb_cv_df.to_csv(os.path.join(MODEL_DIR, 'lgb_cv_tuning.csv'), index=False)
xgb_cv_df.to_csv(os.path.join(MODEL_DIR, 'xgb_cv_tuning.csv'), index=False)

Best LightGBM_L1 CV MAE: 5.2764 ± 1.1164
Best XGBoost_L1 CV MAE:  5.3059 ± 1.1085


## 5. So sánh mô hình trên Validation, Paired Bootstrap & Permutation Importance

In [14]:
ridge_pipe = Pipeline([
    ('prep', make_preprocessor(num_features, cat_features, cat_encoding='onehot')),
    ('model', Ridge(alpha=10.0, random_state=42)),
]).fit(X_train, y_train)

rf = RandomForestRegressor(n_estimators=150, max_depth=14, min_samples_leaf=10, n_jobs=-1, random_state=42).fit(X_train_t, y_train)
xgb_l2 = xgb.XGBRegressor(n_estimators=800, learning_rate=0.05, max_depth=6, subsample=0.8, colsample_bytree=0.8, early_stopping_rounds=50, eval_metric='mae', n_jobs=-1, random_state=42)
xgb_l2.fit(X_train_t, y_train, eval_set=[(X_val_t, y_val)], verbose=False)

lgb_l2 = lgb.LGBMRegressor(objective='regression', n_estimators=800, learning_rate=0.04, num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1, min_child_samples=30, n_jobs=-1, random_state=42, verbose=-1)
lgb_l2.fit(X_train_t, y_train, eval_set=[(X_val_t, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

lgb_l1_tuned = lgb.LGBMRegressor(objective='regression_l1', n_estimators=1500, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1, **best_lgb_params)
lgb_l1_tuned.fit(X_train_t, y_train, eval_set=[(X_val_t, y_val)], callbacks=[lgb.early_stopping(50, verbose=False)])

xgb_l1_tuned = xgb.XGBRegressor(objective='reg:absoluteerror', n_estimators=1500, early_stopping_rounds=50, eval_metric='mae', n_jobs=-1, random_state=42, **best_xgb_params)
xgb_l1_tuned.fit(X_train_t, y_train, eval_set=[(X_val_t, y_val)], verbose=False)

train_median = float(np.median(y_train))
val_preds = {
    'Baseline_Median': np.full_like(y_val, train_median),
    'Baseline_Olist_Estimated': val_df[bench_col].values,
    'Ridge_Regression': ridge_pipe.predict(X_val),
    'Random_Forest': rf.predict(X_val_t),
    'XGBoost_L2': xgb_l2.predict(X_val_t),
    'LightGBM_L2': lgb_l2.predict(X_val_t),
    'LightGBM_L1_Tuned': lgb_l1_tuned.predict(X_val_t),
    'XGBoost_L1_Tuned': xgb_l1_tuned.predict(X_val_t),
}

val_summary = pd.DataFrame([{'Model': k, **eval_metrics(y_val, v)} for k, v in val_preds.items()]).sort_values('MAE')
print('=== VALIDATION LEADERBOARD ===')
print(val_summary.to_string(index=False))

err_xgb = np.abs(y_val - val_preds['XGBoost_L1_Tuned'])
err_lgb = np.abs(y_val - val_preds['LightGBM_L1_Tuned'])
diffs = [float(err_xgb[idx].mean() - err_lgb[idx].mean()) for idx in [np.random.RandomState(42 + i).randint(0, len(y_val), len(y_val)) for i in range(1000)]]
ci_low, ci_high = np.percentile(diffs, [2.5, 97.5])
print(f'\nPaired Bootstrap (XGB_L1 - LGB_L1) Val MAE diff: {np.mean(diffs):+.4f}d | 95% CI: [{ci_low:+.4f}, {ci_high:+.4f}]')

ml_candidates = {
    'Ridge_Regression': ridge_pipe,
    'Random_Forest': Pipeline([('prep', tree_prep), ('model', rf)]),
    'XGBoost_L2': Pipeline([('prep', tree_prep), ('model', xgb_l2)]),
    'LightGBM_L2': Pipeline([('prep', tree_prep), ('model', lgb_l2)]),
    'LightGBM_L1_Tuned': Pipeline([('prep', tree_prep), ('model', lgb_l1_tuned)]),
    'XGBoost_L1_Tuned': Pipeline([('prep', tree_prep), ('model', xgb_l1_tuned)]),
}
best_name = val_summary[val_summary['Model'].isin(ml_candidates.keys())].iloc[0]['Model']
best_pipe = ml_candidates[best_name]
print(f'Locked Best Model (by Val MAE): {best_name}')

perm = permutation_importance(best_pipe, X_val, y_val, scoring='neg_mean_absolute_error', n_repeats=5, random_state=42, n_jobs=-1)
perm_df = pd.DataFrame({'Feature': all_features, 'Perm_MAE_Increase': perm.importances_mean, 'Std': perm.importances_std}).sort_values('Perm_MAE_Increase', ascending=False)
perm_df.to_csv(os.path.join(MODEL_DIR, 'permutation_importance.csv'), index=False)
print('\nTop 10 Permutation Importance (Val set):')
print(perm_df.head(10).to_string(index=False))

=== VALIDATION LEADERBOARD ===
                   Model       MAE      RMSE        R2  Within_3d_%  Within_5d_%
       LightGBM_L1_Tuned  4.094787  6.205046  0.353195    50.234969    71.962681
        XGBoost_L1_Tuned  4.134686  6.209887  0.352186    49.232896    72.114720
             LightGBM_L2  4.576119  6.382351  0.315703    40.262612    65.694540
           Random_Forest  4.706943  6.500827  0.290062    39.329648    63.773324
        Ridge_Regression  4.727679  6.607077  0.266666    41.292329    63.386317
              XGBoost_L2  4.753914  6.566147  0.275724    38.272287    64.077402
         Baseline_Median  5.693889  7.741468 -0.006768    29.018659    51.886662
Baseline_Olist_Estimated 14.910214 17.266384 -4.008247     5.183138     9.633725

Paired Bootstrap (XGB_L1 - LGB_L1) Val MAE diff: +0.0400d | 95% CI: [+0.0288, +0.0522]
Locked Best Model (by Val MAE): LightGBM_L1_Tuned

Top 10 Permutation Importance (Val set):
             Feature  Perm_MAE_Increase      Std
        hav

## 6. Đánh giá cuối cùng trên Test Holdout (1 lần duy nhất) & Error Analysis + Top 5 Worst Errors

In [15]:
X_test_t = tree_prep.transform(X_test)
test_preds = {
    'Baseline_Median': np.full_like(y_test, train_median),
    'Baseline_Olist_Estimated': test_df[bench_col].values,
    'Ridge_Regression': ridge_pipe.predict(X_test),
    'Random_Forest': rf.predict(X_test_t),
    'XGBoost_L2': xgb_l2.predict(X_test_t),
    'LightGBM_L2': lgb_l2.predict(X_test_t),
    'LightGBM_L1_Tuned': lgb_l1_tuned.predict(X_test_t),
    'XGBoost_L1_Tuned': xgb_l1_tuned.predict(X_test_t),
}

final_rows = []
for name in val_summary['Model']:
    vm = eval_metrics(y_val, val_preds[name])
    tm = eval_metrics(y_test, test_preds[name])
    final_rows.append({
        'Model': name, 'Val_MAE': vm['MAE'], 'Val_RMSE': vm['RMSE'],
        'Test_MAE': tm['MAE'], 'Test_RMSE': tm['RMSE'],
        'Test_R2': tm['R2'], 'Test_Within_3d_%': tm['Within_3d_%'], 'Test_Within_5d_%': tm['Within_5d_%'],
    })

results_df = pd.DataFrame(final_rows).sort_values('Test_MAE')
results_df.to_csv(os.path.join(MODEL_DIR, 'model_comparison.csv'), index=False)
joblib.dump(best_pipe, os.path.join(MODEL_DIR, 'best_model.joblib'))
print('=== FINAL HOLDOUT TEST RESULTS ===')
print(results_df.to_string(index=False))

best_val_pred = val_preds[best_name]
best_test_pred = test_preds[best_name]
med_test_pred = test_preds['Baseline_Median']

ea_df = test_df.copy()
ea_df['pred'] = best_test_pred
ea_df['abs_err'] = np.abs(y_test - best_test_pred)
ea_df['sq_err'] = (y_test - best_test_pred) ** 2
ea_df['med_abs_err'] = np.abs(y_test - med_test_pred)
ea_df['test_month'] = pd.to_datetime(ea_df['order_purchase_timestamp']).dt.to_period('M').astype(str)
ea_df['dist_bin'] = pd.cut(ea_df['haversine_km'], bins=[-1, 50, 200, 500, 1000, 10000], labels=['<50km', '50-200km', '200-500km', '500-1000km', '>1000km'])
ea_df['weight_bin'] = pd.qcut(ea_df['total_weight_g'], q=4, labels=['Q1_Light', 'Q2_MedLow', 'Q3_MedHigh', 'Q4_Heavy'], duplicates='drop')
ea_df['seller_group'] = np.where(ea_df['num_unique_sellers'] == 1, '1_Seller', 'Multi_Seller')
ea_df['speed_group'] = pd.cut(ea_df['delivery_time_days'], bins=[0, 7, 15, 30, 500], labels=['Fast(<=7d)', 'Medium(7-15d)', 'Slow(15-30d)', 'VerySlow(>30d)'])

group_tables = []
for gcol in ['dist_bin', 'weight_bin', 'seller_group', 'speed_group', 'test_month', 'customer_state']:
    grp = ea_df.groupby(gcol, observed=False).agg(
        n=('order_id', 'count'), MAE=('abs_err', 'mean'), MedAE=('abs_err', 'median'),
        RMSE=('sq_err', lambda s: np.sqrt(s.mean())), Baseline_Median_MAE=('med_abs_err', 'mean'),
    ).reset_index().rename(columns={gcol: 'Group'})
    grp.insert(0, 'Dimension', gcol)
    if gcol == 'customer_state':
        grp = grp[grp['n'] >= 100].sort_values('MAE', ascending=False)
    group_tables.append(grp)

error_analysis_df = pd.concat(group_tables, ignore_index=True)
error_analysis_df.to_csv(os.path.join(MODEL_DIR, 'error_analysis.csv'), index=False)

# Top 5 Worst Errors (P1.7)
worst_5 = ea_df.sort_values('abs_err', ascending=False)[['order_id', 'order_purchase_timestamp', 'customer_state', 'primary_seller_state', 'haversine_km', 'total_weight_g', 'delivery_time_days', 'pred', 'abs_err']].head(5)
worst_5.to_csv(os.path.join(MODEL_DIR, 'worst_5_errors.csv'), index=False)
print('\nTop 5 Worst Errors on Test:')
print(worst_5.to_string(index=False))

=== FINAL HOLDOUT TEST RESULTS ===
                   Model   Val_MAE  Val_RMSE  Test_MAE  Test_RMSE   Test_R2  Test_Within_3d_%  Test_Within_5d_%
       LightGBM_L1_Tuned  4.094787  6.205046  3.880062   5.267827  0.069645         47.591735         71.156105
        XGBoost_L1_Tuned  4.134686  6.209887  3.913384   5.299619  0.058382         46.831594         71.764218
        Ridge_Regression  4.727679  6.607077  4.337185   5.900464 -0.167234         45.808859         66.346486
             LightGBM_L2  4.576119  6.382351  4.738992   6.000943 -0.207326         35.477852         60.320641
              XGBoost_L2  4.753914  6.566147  4.824564   6.128312 -0.259120         34.959574         60.058047
           Random_Forest  4.706943  6.500827  4.909550   6.165973 -0.274644         33.992122         58.088591
         Baseline_Median  5.693889  7.741468  5.086576   6.283504 -0.323699         27.717504         52.069657
Baseline_Olist_Estimated 14.910214 17.266384 10.722452  13.112371 -4.

## 7. Xuất các biểu đồ báo cáo (bao gồm MAE theo tháng trên Val & Test)

In [16]:
# 1. Model Comparison
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
plot_df = results_df.sort_values('Test_MAE')
sns.barplot(data=plot_df, x='Test_MAE', y='Model', ax=axes[0], palette='Blues_d')
axes[0].set_title('Test MAE (ngày) — Thấp hơn là tốt hơn')
sns.barplot(data=plot_df, x='Test_RMSE', y='Model', ax=axes[1], palette='Greens_d')
axes[1].set_title('Test RMSE (ngày) — Thấp hơn là tốt hơn')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '01_model_comparison.png'), bbox_inches='tight')
plt.close()

# 2. Permutation Importance
plt.figure(figsize=(10, 6))
sns.barplot(data=perm_df.head(15), x='Perm_MAE_Increase', y='Feature', palette='viridis')
plt.title(f'Top 15 Permutation Importance trên Validation ({best_name})')
plt.xlabel('Mức tăng MAE khi xáo trộn feature (ngày)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '02_feature_importance.png'), bbox_inches='tight')
plt.close()

# 3. Actual vs Predicted & Residuals
residuals = y_test - best_test_pred
fig, axes = plt.subplots(1, 2, figsize=(13, 5))
s_idx = np.random.RandomState(42).choice(len(y_test), size=min(4000, len(y_test)), replace=False)
axes[0].scatter(y_test[s_idx], best_test_pred[s_idx], alpha=0.25, s=12, color='#2b8cbe')
max_val = min(45, float(np.percentile(y_test, 99)))
axes[0].plot([0, max_val], [0, max_val], 'r--', lw=2, label='Ideal (y = x)')
axes[0].set_xlim(0, max_val); axes[0].set_ylim(0, max_val)
axes[0].set_title(f'Actual vs Predicted ({best_name})'); axes[0].legend()
sns.histplot(np.clip(residuals, -20, 20), bins=50, kde=True, ax=axes[1], color='#31a354')
axes[1].axvline(0, color='red', linestyle='--')
axes[1].set_title('Phân phối phần dư (Residuals, clipped ±20d)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '03_actual_vs_predicted.png'), bbox_inches='tight')
plt.close()

# 4. Error by State
st_df = error_analysis_df[error_analysis_df['Dimension'] == 'customer_state'].head(10)
st_melt = st_df.melt(id_vars='Group', value_vars=['Baseline_Median_MAE', 'MAE'], var_name='Method', value_name='MAE_Val')
st_melt['Method'] = st_melt['Method'].map({'Baseline_Median_MAE': 'Baseline Median', 'MAE': best_name})
plt.figure(figsize=(11, 5))
sns.barplot(data=st_melt, x='Group', y='MAE_Val', hue='Method', palette='Set2')
plt.title('So sánh MAE theo Top 10 Bang có sai số cao nhất (n >= 100)')
plt.xlabel('Bang (customer_state)'); plt.ylabel('MAE (ngày)')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '04_error_by_state.png'), bbox_inches='tight')
plt.close()

# 5. Monthly MAE on Val & Test (P0.4)
plt.figure(figsize=(10, 4.5))
for split_name, d_df, y_true_s, y_pred_s in [('Validation', val_df, y_val, best_val_pred), ('Test', test_df, y_test, best_test_pred)]:
    m_tmp = pd.DataFrame({'month': pd.to_datetime(d_df['order_purchase_timestamp']).dt.to_period('M').astype(str), 'abs_err': np.abs(y_true_s - y_pred_s)})
    m_grp = m_tmp.groupby('month')['abs_err'].agg(['count', 'mean']).reset_index()
    m_grp = m_grp[m_grp['count'] >= 50]
    plt.plot(m_grp['month'], m_grp['mean'], marker='o', linewidth=2, label=split_name)
plt.title(f'MAE theo tháng trên Validation & Test ({best_name})')
plt.xlabel('Tháng đặt hàng'); plt.ylabel('MAE (ngày)'); plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, '05_monthly_error_val_test.png'), bbox_inches='tight')
plt.close()

## 8. Thí nghiệm bổ sung: Sensitivity Analysis (Cutoff & Outlier Cap) + Quantile Regression (P10–P90)

In [17]:
sens_rows = []
# Baseline Full (uncapped)
m_full_v = eval_metrics(y_val, best_val_pred)
m_full_t = eval_metrics(y_test, best_test_pred)
sens_rows.append({'Experiment': 'Full_Original', 'Val_MAE': m_full_v['MAE'], 'Val_RMSE': m_full_v['RMSE'], 'Val_R2': m_full_v['R2'], 'Test_MAE': m_full_t['MAE'], 'Test_RMSE': m_full_t['RMSE'], 'Test_R2': m_full_t['R2']})

# 8.1 Right-censoring Cutoff (<= 2018-06-30)
c_tr = pd.read_csv(os.path.join(PROC_DIR, 'cutoff_train.csv'))
c_va = pd.read_csv(os.path.join(PROC_DIR, 'cutoff_val.csv'))
c_te = pd.read_csv(os.path.join(PROC_DIR, 'cutoff_test.csv'))
c_prep = make_preprocessor(num_features, cat_features, cat_encoding=best_tree_enc)
Xc_tr = c_prep.fit_transform(c_tr[all_features])
Xc_va = c_prep.transform(c_va[all_features])
Xc_te = c_prep.transform(c_te[all_features])
m_cut = lgb.LGBMRegressor(objective='regression_l1', n_estimators=1500, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1, **best_lgb_params)
m_cut.fit(Xc_tr, c_tr[target_col].values, eval_set=[(Xc_va, c_va[target_col].values)], callbacks=[lgb.early_stopping(50, verbose=False)])
cv_m = eval_metrics(c_va[target_col].values, m_cut.predict(Xc_va))
ct_m = eval_metrics(c_te[target_col].values, m_cut.predict(Xc_te))
sens_rows.append({'Experiment': 'Cutoff_2018_06_30', 'Val_MAE': cv_m['MAE'], 'Val_RMSE': cv_m['RMSE'], 'Val_R2': cv_m['R2'], 'Test_MAE': ct_m['MAE'], 'Test_RMSE': ct_m['RMSE'], 'Test_R2': ct_m['R2']})

# 8.2 Outlier Cap Sensitivity (CAP @ p99 của Train)
cap_val = float(np.quantile(y_train, 0.99))
y_tr_cap = np.clip(y_train, None, cap_val)
m_cap = lgb.LGBMRegressor(objective='regression_l1', n_estimators=1500, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1, **best_lgb_params)
m_cap.fit(X_train_t, y_tr_cap, eval_set=[(X_val_t, np.clip(y_val, None, cap_val))], callbacks=[lgb.early_stopping(50, verbose=False)])
cap_v = eval_metrics(y_val, m_cap.predict(X_val_t))
cap_t = eval_metrics(y_test, m_cap.predict(X_test_t))
sens_rows.append({'Experiment': f'Outlier_Cap_p99_{cap_val:.1f}d', 'Val_MAE': cap_v['MAE'], 'Val_RMSE': cap_v['RMSE'], 'Val_R2': cap_v['R2'], 'Test_MAE': cap_t['MAE'], 'Test_RMSE': cap_t['RMSE'], 'Test_R2': cap_t['R2']})

sens_df = pd.DataFrame(sens_rows)
sens_df.to_csv(os.path.join(MODEL_DIR, 'sensitivity_results.csv'), index=False)
print('=== SENSITIVITY ANALYSIS ===')
print(sens_df.to_string(index=False))

# 8.3 Quantile Regression (P10 - P90 prediction interval)
q10 = lgb.LGBMRegressor(objective='quantile', alpha=0.10, n_estimators=800, learning_rate=0.04, num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1).fit(X_train_t, y_train)
q90 = lgb.LGBMRegressor(objective='quantile', alpha=0.90, n_estimators=800, learning_rate=0.04, num_leaves=63, colsample_bytree=0.8, subsample=0.8, subsample_freq=1, n_jobs=-1, random_state=42, verbose=-1).fit(X_train_t, y_train)
p10_test = q10.predict(X_test_t)
p90_test = np.maximum(q90.predict(X_test_t), p10_test)
coverage = float(((y_test >= p10_test) & (y_test <= p90_test)).mean() * 100.0)
mean_width = float((p90_test - p10_test).mean())
q_df = pd.DataFrame([{'Target_Coverage_%': 80.0, 'Test_Empirical_Coverage_%': coverage, 'Mean_Interval_Width_Days': mean_width, 'Median_Width_Days': float(np.median(p90_test - p10_test))}])
q_df.to_csv(os.path.join(MODEL_DIR, 'quantile_summary.csv'), index=False)
print('\n=== QUANTILE REGRESSION (P10 - P90) ===')
print(q_df.to_string(index=False))

=== SENSITIVITY ANALYSIS ===
           Experiment  Val_MAE  Val_RMSE   Val_R2  Test_MAE  Test_RMSE  Test_R2
        Full_Original 4.094787  6.205046 0.353195  3.880062   5.267827 0.069645
    Cutoff_2018_06_30 5.199917  7.814916 0.379822  4.345482   6.318080 0.284642
Outlier_Cap_p99_48.9d 4.097972  6.206641 0.352863  3.882852   5.271014 0.068519

=== QUANTILE REGRESSION (P10 - P90) ===
 Target_Coverage_%  Test_Empirical_Coverage_%  Mean_Interval_Width_Days  Median_Width_Days
              80.0                  54.965103                  9.589917           8.786041
